In [1]:
%pip install -q youtube-transcript-api langchain langchain-community langchain-google-genai faiss-cpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass

# Generate a free key at Google AI Studio and paste it in the prompt
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API Key: ")

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")
if api_key:
    print(f"Key loaded successfully: {api_key[:5]}...")
else:
    print("Failed to load key. Check your .env file.")

# The rest of your imports go here
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

Key loaded successfully: AQ.Ab...


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15412\386611848.py:16: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
video_id = "Gfr50f6ZBvo" # Lex Fridman podcast example

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(f"Successfully loaded {len(transcript)} characters.")
except TranscriptsDisabled:
    print("No captions available for this video.")

Successfully loaded 133836 characters.


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)

chunks = text_splitter.create_documents([transcript])
print(f"Split into {len(chunks)} chunks.")

Split into 149 chunks.


In [6]:
import os
import time

# Initialize the embedding model
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

print(f"Total chunks to process: {len(chunks)}")
print("Building vector database in batches to respect free tier limits...")

# 1. Initialize the vector database with the first 90 chunks
batch_size = 90
vector_store = FAISS.from_documents(chunks[:batch_size], embeddings)
print(f"Processed first {batch_size} chunks.")

# 2. Loop through any remaining chunks in batches, pausing in between
for i in range(batch_size, len(chunks), batch_size):
    print("Waiting 60 seconds for API quota to reset...")
    time.sleep(60) # Pause execution for 1 minute
    
    batch = chunks[i : i + batch_size]
    vector_store.add_documents(batch)
    print(f"Processed next {len(batch)} chunks.")

Total chunks to process: 149
Building vector database in batches to respect free tier limits...
Processed first 90 chunks.
Waiting 60 seconds for API quota to reset...
Processed next 59 chunks.


In [7]:
retriever = vector_store.as_retriever(search_kwargs={'k': 4})
question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"

# Manually fetch the 4 most relevant chunks
retrieved_docs = retriever.invoke(question)
retrieved_docs

[Document(id='c5438f37-5ab2-4411-85ce-1fc60ce0f952', metadata={}, page_content="speaking of which you have a paper on nuclear fusion uh magnetic control of tokamak plasmas to deep reinforcement learning so you uh you're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this work and uh can ai eventually solve nuclear fusion it's been very fun last year or two and very productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of areas of science that i would like to i think could be very transformative if we helped accelerate and uh really interesting problems scientific challenges in of themselves this is energy so energy yes exactly so energy and climate so we talked about disease and biology as being one of the biggest places i think ai can help with i think energy and climate uh is another one so maybe they would be my top two um and fusion is one one area 

In [31]:
# Manual Augmentation
import os

os.environ["GOOGLE_API_TRANSPORT"] = "rest"
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    timeout=300, # Give the model a full minute to process the transcript
    max_retries=0  
)

prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables=['context', 'question']
)

# Manually stitch the retrieved text chunks together
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

# Manually insert the text and question into the prompt template
final_prompt = prompt.invoke({"context": context_text, "question": question})
print(final_prompt)

text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      speaking of which you have a paper on nuclear fusion uh magnetic control of tokamak plasmas to deep reinforcement learning so you uh you're seeking to solve nuclear fusion with deep rl so it's doing control of high temperature plasmas can you explain this work and uh can ai eventually solve nuclear fusion it's been very fun last year or two and very productive because we've been taking off a lot of my dream projects if you like of things that i've collected over the years of areas of science that i would like to i think could be very transformative if we helped accelerate and uh really interesting problems scientific challenges in of themselves this is energy so energy yes exactly so energy and climate so we talked about disease and biology as being one of the biggest places i think ai can help with i think energy a

In [ ]:
# Manually pass the filled-out prompt to the LLM
answer = llm.invoke(final_prompt)
print(answer.content)

In [33]:


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

parser = StrOutputParser()

# Tie it all together
main_chain = parallel_chain | prompt | llm | parser

In [35]:
# Now you can just pass a question, and the chain does the retrieval and generation automatically!
chain_answer = main_chain.invoke('who is Demis')
print(chain_answer)

Based on the provided transcript, Demis (Demis Hassabis) is the CEO and co-founder of DeepMind. He is described as one of the most brilliant and impactful humans in the history of artificial intelligence, science, and engineering in general.
